# Telco customer churn – exploratory data analysis

This notebook maps the behavioural and contractual signals that precede customer departure. Every pattern found here will surface directly in the SHAP waterfall charts shown in the Streamlit app — understanding *why* each feature matters now makes the model's predictions interpretable end-to-end.

To prevent data leakage, the dataset is split before any analysis. All exploration is performed on the training set only.

In [14]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from scipy import stats
from scipy.stats import pointbiserialr
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')
pio.templates.default = 'plotly_white'

CHURN  = "#974243"
RETAIN = "#445F8E"
ACCENT = "#17530E"
COLOR_MAP = {'Yes': CHURN, 'No': RETAIN}

## Data loading and preparation

`TotalCharges` is stored as text because zero-tenure customers have a blank entry instead of `0.00`; coercing to numeric turns those blanks into NaN. `customerID` is a row key with no predictive signal and is dropped immediately. The 80/20 stratified split keeps the churn ratio consistent in both partitions.

In [ ]:
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.drop(columns=['customerID'], inplace=True)

X = df.drop('Churn', axis=1)
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

df_train = X_train.copy()
df_train['Churn'] = y_train.values

# OVERALL_CHURN is reused as a reference line in every churn-rate chart
OVERALL_CHURN = (df_train['Churn'] == 'Yes').mean() * 100

print(f'Training set : {df_train.shape[0]} rows x {df_train.shape[1]} columns')
print(f'Test set : {X_test.shape[0]} rows x {X_test.shape[1]} columns')
print(f'Overall churn rate  : {OVERALL_CHURN:.1f}%')

Training set        : 5,634 rows x 20 columns
Test set            : 1,409 rows x 19 columns
Overall churn rate  : 26.5%


## Dataset overview

In [16]:
df_train.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
3738,Male,0,No,No,35,No,No phone service,DSL,No,No,Yes,No,Yes,Yes,Month-to-month,No,Electronic check,49.20,1701.65,No
3151,Male,0,Yes,Yes,15,Yes,No,Fiber optic,Yes,No,No,No,No,No,Month-to-month,No,Mailed check,75.10,1151.55,No
4860,Male,0,Yes,Yes,13,No,No phone service,DSL,Yes,Yes,No,Yes,No,No,Two year,No,Mailed check,40.55,590.35,No
3867,Female,0,Yes,No,26,Yes,No,DSL,No,Yes,Yes,No,Yes,Yes,Two year,Yes,Credit card (automatic),73.50,1905.70,No
3810,Male,0,Yes,Yes,1,Yes,No,DSL,No,No,No,No,No,No,Month-to-month,No,Electronic check,44.55,44.55,No


In [ ]:
df_train.describe(include='all').T.round(2)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
gender,5634,2,Male,2833,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SeniorCitizen,5634.0,NaN,NaN,NaN,0.163294,0.369667,0.0,0.0,0.0,0.0,1.0
Partner,5634,2,No,2905,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dependents,5634,2,No,3955,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure,5634.0,NaN,NaN,NaN,32.485091,24.568744,0.0,9.0,29.0,55.0,72.0
PhoneService,5634,2,Yes,5075,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MultipleLines,5634,3,No,2685,NaN,NaN,NaN,NaN,NaN,NaN,NaN
InternetService,5634,3,Fiber optic,2483,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OnlineSecurity,5634,3,No,2797,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OnlineBackup,5634,3,No,2442,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
# The NaNs in TotalCharges belong to customers with tenure = 0 who have not yet been billed
missing = df_train.isnull().sum().rename('missing_count').to_frame()
missing['pct_of_rows'] = (missing['missing_count'] / len(df_train) * 100).round(2)
missing[missing['missing_count'] > 0]

,missing_count,pct_of_rows
TotalCharges,8,0.14


## Class imbalance

At ~26 % churn, the dataset is moderately imbalanced. A naive model predicting "no churn" for everyone would achieve 74 % accuracy while catching zero churners — accuracy is therefore a misleading metric. We will use ROC-AUC, precision-recall, and class-weighted training instead.

In [ ]:
counts = df_train['Churn'].value_counts().reset_index()
counts.columns = ['Churn', 'Count']
rate = counts.loc[counts['Churn'] == 'Yes', 'Count'].values[0] / counts['Count'].sum()

fig = go.Figure(go.Pie(
    labels=counts['Churn'], values=counts['Count'], hole=0.58,
    marker=dict(colors=[RETAIN, CHURN], line=dict(color='white', width=2)),
    textinfo='label+percent+value', textfont_size=13, direction='clockwise',
    sort=False, hovertemplate='%{label}: %{value:,} customers<extra></extra>',
))
fig.update_layout(
    title_text='Class imbalance',
    annotations=[dict(
        text=f'<b>{rate:.1%}</b><br>churn rate',
        x=0.5, y=0.5, font_size=16, showarrow=False, align='center',
    )],
    height=450,
)
fig.show()

## Tenure cohort analysis

Tenure is the customer's lifespan in months and is arguably the most important time dimension in churn analysis. New customers are in a trial mindset with low switching costs; long-tenured customers have absorbed those costs and built habits around the service. We bucket tenure into four periods to quantify exactly *when* attrition peaks — this directly informs how the model should weight short-tenure customers and what the SHAP waterfall will emphasise for a month-0 prediction.

In [ ]:
df_train['tenure_bucket'] = pd.cut(
    df_train['tenure'], bins=[0, 12, 24, 48, 72], labels=['0–12 m', '13–24 m', '25–48 m', '49–72 m'],
)

cohort = (
    df_train.groupby('tenure_bucket', observed=True)
    .agg(n_customers=('Churn', 'size'), n_churned=('Churn', lambda x: (x == 'Yes').sum()))
    .assign(churn_rate=lambda d: d['n_churned'] / d['n_customers'] * 100)
    .reset_index()
)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Churn rate per cohort', 'Customer volume per cohort'),
    horizontal_spacing=0.12,
)

# Color each bar relative to overall rate: red if above, blue if below
bar_colors = [CHURN if v > OVERALL_CHURN else RETAIN for v in cohort['churn_rate']]

fig.add_trace(go.Bar(
    x=cohort['tenure_bucket'].astype(str),
    y=cohort['churn_rate'],
    marker_color=bar_colors,
    text=cohort['churn_rate'].map('{:.1f}%'.format),
    textposition='outside',
    hovertemplate='%{x}: %{y:.1f}% churn<extra></extra>',
), row=1, col=1)

fig.add_trace(go.Bar(
    x=cohort['tenure_bucket'].astype(str),
    y=cohort['n_customers'],
    marker_color='#A5B4C8',
    text=cohort['n_customers'],
    textposition='outside',
    hovertemplate='%{x}: %{y:,} customers<extra></extra>',
), row=1, col=2)

fig.add_hline(
    y=OVERALL_CHURN, line_dash='dash', line_color='gray', line_width=1, row=1, col=1,
    annotation_text=f'Overall {OVERALL_CHURN:.1f}%', annotation_position='top right',
)
fig.update_yaxes(title_text='Churn rate (%)', row=1, col=1)
fig.update_yaxes(title_text='Customer count', row=1, col=2)
fig.update_layout(title_text='Tenure cohort analysis', showlegend=False, height=460)
fig.show()

## Churn rate by categorical feature

Raw counts hide the actual risk because category sizes differ. Each bar shows the percentage of customers in that category who churned, sorted lowest to highest. Bars are coloured **red** when above the overall rate (dashed line) and **blue** when below. This is the key reference chart for building intuition before interpreting SHAP: every red bar identifies a group where the model should — and will — assign a positive SHAP contribution.

In [ ]:
cat_cols = sorted(
    df_train.select_dtypes(include='object').columns.difference(['Churn']).tolist()
)

n_cols = 3
n_rows = -(-len(cat_cols) // n_cols)

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=cat_cols,
    horizontal_spacing=0.18,
    vertical_spacing=0.07,
)

for idx, col in enumerate(cat_cols):
    r = idx // n_cols + 1
    c = idx % n_cols + 1
    rate = (
        df_train.groupby(col)['Churn']
        .apply(lambda x: (x == 'Yes').mean() * 100)
        .rename('ChurnRate')
        .reset_index()
        .sort_values('ChurnRate', ascending=True)
    )
    colors = [CHURN if v > OVERALL_CHURN else RETAIN for v in rate['ChurnRate']]
    fig.add_trace(go.Bar(
        x=rate['ChurnRate'],
        y=rate[col],
        orientation='h',
        marker_color=colors,
        text=rate['ChurnRate'].map('{:.1f}%'.format),
        textposition='outside',
        hovertemplate='%{y}: %{x:.1f}%<extra></extra>',
    ), row=r, col=c)
    fig.add_vline(x=OVERALL_CHURN, line_dash='dash', line_color='gray', line_width=1, row=r, col=c)
    fig.update_xaxes(title_text='Churn rate (%)', row=r, col=c)

fig.update_layout(
    height=295 * n_rows,
    title_text='Churn rate by categorical feature  |  red = above-average risk  |  dashed = overall rate',
    showlegend=False,
)
fig.show()

## The service bundle effect

Telecom companies use value-added services as retention anchors. Each add-on raises the perceived cost of switching: cancelling the subscription means losing not just the connection but also cloud backup, device protection, and tech support — all at once. We count active subscriptions per customer (0–6) and check whether churn falls monotonically as bundle depth increases. If it does, `n_services` is effectively a composite loyalty score and will be a powerful engineered feature.

In [ ]:
service_cols = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies',
]
# 'No internet service' and 'No' both count as 0
df_train['n_services'] = df_train[service_cols].apply(lambda row: (row == 'Yes').sum(), axis=1)

bundle = (
    df_train.groupby('n_services')
    .agg(n_customers=('Churn', 'size'), churn_rate=('Churn', lambda x: (x == 'Yes').mean() * 100))
    .reset_index()
)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Churn rate by service count', 'Customer distribution by service count'),
    horizontal_spacing=0.12,
)

fig.add_trace(go.Bar(
    x=bundle['n_services'],
    y=bundle['churn_rate'],
    marker_color=[CHURN if v > OVERALL_CHURN else RETAIN for v in bundle['churn_rate']],
    text=bundle['churn_rate'].map('{:.1f}%'.format),
    textposition='outside',
    hovertemplate='%{x} services: %{y:.1f}% churn<extra></extra>',
), row=1, col=1)

fig.add_trace(go.Bar(
    x=bundle['n_services'],
    y=bundle['n_customers'],
    marker_color='#A5B4C8',
    text=bundle['n_customers'],
    textposition='outside',
    hovertemplate='%{x} services: %{y:,} customers<extra></extra>',
), row=1, col=2)

fig.add_hline(
    y=OVERALL_CHURN, line_dash='dash', line_color='gray', line_width=1, row=1, col=1,
    annotation_text=f'Overall {OVERALL_CHURN:.1f}%', annotation_position='top right',
)
fig.update_xaxes(title_text='Number of add-on services', dtick=1)
fig.update_yaxes(title_text='Churn rate (%)', row=1, col=1)
fig.update_layout(title_text='Service bundle effect on retention', showlegend=False, height=460)
fig.show()

## Risk segments: contract type × internet service

Contract type and internet service are the two strongest categorical predictors found so far. Crossing them produces a risk matrix that reveals the **compound effect**: a month-to-month fibre-optic customer faces both the absence of commitment and a higher bill — the worst combination for retention. This interaction is important context when reading SHAP interaction values later: the model will likely learn it implicitly, and the SHAP chart for a fibre-optic month-to-month customer will show *both* features pushing the probability upward.

In [ ]:
pivot = (
    df_train.groupby(['Contract', 'InternetService'])['Churn']
    .apply(lambda x: round((x == 'Yes').mean() * 100, 1))
    .unstack()
)

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale='RdYlGn_r',
    zmin=0, zmax=65,
    text=pivot.values,
    texttemplate='%{text:.1f}%',
    textfont_size=15,
    hovertemplate='Contract: %{y}<br>Internet: %{x}<br>Churn rate: %{z:.1f}%<extra></extra>',
    colorbar=dict(title='Churn rate (%)'),
))
fig.update_layout(
    title_text='Churn rate (%) by contract type and internet service',
    xaxis_title='Internet service',
    yaxis_title='Contract type',
    height=380,
)
fig.show()

## Numerical feature distributions

`tenure`, `MonthlyCharges` and `TotalCharges` are the three continuous variables. The histograms show how the distribution shapes differ between churned and retained customers; box plots quantify the median and interquartile range. The degree of separation between the two groups is a direct proxy for how discriminative each feature will be for the classifier — and how prominently it will appear in the SHAP summary plot.

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=num_cols + ['', '', ''],
    vertical_spacing=0.14,
    horizontal_spacing=0.09,
)

for i, col in enumerate(num_cols, 1):
    for churn_val in ['No', 'Yes']:
        data = df_train.loc[df_train['Churn'] == churn_val, col].dropna()
        fig.add_trace(go.Histogram(
            x=data,
            name=f'Churn = {churn_val}',
            marker_color=COLOR_MAP[churn_val],
            opacity=0.72,
            legendgroup=churn_val,
            showlegend=(i == 1),
            hovertemplate='%{x}<br>Count: %{y}<extra></extra>',
        ), row=1, col=i)
        fig.add_trace(go.Box(
            y=data,
            name=f'Churn = {churn_val}',
            marker_color=COLOR_MAP[churn_val],
            legendgroup=churn_val,
            showlegend=False,
            boxmean=True,
        ), row=2, col=i)

fig.update_layout(
    barmode='overlay',
    height=680,
    title_text='Numerical feature distributions by churn status',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, x=0),
)
fig.show()

## Monthly charges by contract type

Fibre-optic subscribers pay significantly more than DSL customers. When we layer contract type on top, the picture sharpens: month-to-month customers paying high monthly charges with no lock-in are the most flight-prone segment. This creates a core business tension — the highest-revenue customers are also the ones most likely to churn. For the SHAP chart, a high `MonthlyCharges` value will consistently push the churn probability upward, especially when combined with a month-to-month contract.

In [ ]:
fig = px.violin(
    df_train,
    x='Contract',
    y='MonthlyCharges',
    color='Churn',
    color_discrete_map=COLOR_MAP,
    box=True,
    points=False,
    violinmode='overlay',
    category_orders={'Contract': ['Month-to-month', 'One year', 'Two year']},
    labels={'MonthlyCharges': 'Monthly charges (USD)', 'Contract': 'Contract type'},
    title='Monthly charges by contract type and churn status',
    height=490,
)
fig.update_traces(opacity=0.75)
fig.show()

## Senior citizens: a vulnerable segment

`SeniorCitizen` is a binary flag (0/1) that can easily be overlooked as a weak predictor. The data tells a different story: senior customers churn at nearly twice the rate of non-seniors. The gap is widest on month-to-month contracts and narrows — but does not disappear — on longer commitments. The mechanism is likely a combination of factors: seniors may be more price-sensitive on fixed incomes, less comfortable escalating service complaints, and more susceptible to competitor offers delivered by phone or door-to-door. In the SHAP chart, `SeniorCitizen = 1` will push the probability consistently upward.

In [ ]:
senior_churn = (
    df_train.groupby(['SeniorCitizen', 'Contract'])['Churn']
    .apply(lambda x: (x == 'Yes').mean() * 100)
    .reset_index(name='ChurnRate')
)
senior_churn['SeniorCitizen'] = senior_churn['SeniorCitizen'].map({0: 'Non-senior', 1: 'Senior citizen'})

fig = px.bar(
    senior_churn,
    x='Contract',
    y='ChurnRate',
    color='SeniorCitizen',
    barmode='group',
    color_discrete_sequence=[RETAIN, CHURN],
    text=senior_churn['ChurnRate'].map('{:.1f}%'.format),
    category_orders={'Contract': ['Month-to-month', 'One year', 'Two year']},
    labels={'ChurnRate': 'Churn rate (%)', 'Contract': 'Contract type', 'SeniorCitizen': ''},
    title='Churn rate by contract type and senior-citizen status',
    height=460,
)
fig.update_traces(textposition='outside')
fig.add_hline(
    y=OVERALL_CHURN, line_dash='dash', line_color='gray', line_width=1,
    annotation_text=f'Overall {OVERALL_CHURN:.1f}%', annotation_position='top right',
)
fig.show()

## Payment method and paperless billing

Payment method is a proxy for the customer's level of engagement and automation. Customers who set up automatic payments (credit card or bank transfer) have actively committed to continuity; those paying by electronic check each month are re-evaluating the relationship every billing cycle. Paperless billing amplifies this: paperless customers receive digital invoices and are more exposed to price comparisons with competitors. The combination `PaperlessBilling=Yes` + `PaymentMethod=Electronic check` consistently marks the highest-risk payment profile.

In [ ]:
pay_pivot = (
    df_train.groupby(['PaymentMethod', 'PaperlessBilling'])['Churn']
    .apply(lambda x: round((x == 'Yes').mean() * 100, 1))
    .unstack()
)

fig = go.Figure(go.Heatmap(
    z=pay_pivot.values,
    x=pay_pivot.columns.tolist(),
    y=pay_pivot.index.tolist(),
    colorscale='RdYlGn_r',
    zmin=0, zmax=60,
    text=pay_pivot.values,
    texttemplate='%{text:.1f}%',
    textfont_size=14,
    hovertemplate='Payment: %{y}<br>Paperless: %{x}<br>Churn rate: %{z:.1f}%<extra></extra>',
    colorbar=dict(title='Churn rate (%)'),
))
fig.update_layout(
    title_text='Churn rate (%) by payment method and paperless billing',
    xaxis_title='Paperless billing',
    yaxis_title='Payment method',
    height=380,
)
fig.show()

## Correlation matrix

`TotalCharges` is mathematically close to `tenure × MonthlyCharges`, so its correlation with both is strong. The near-perfect correlation between `TotalCharges` and `tenure` (r ≈ 0.83) means these two features carry largely redundant information. Keeping both during modelling may dilute SHAP values by spreading importance across correlated features — the preprocessing step should address this before training.

In [ ]:
corr = df_train[num_cols].corr().round(2)

fig = go.Figure(go.Heatmap(
    z=corr.values,
    x=corr.columns.tolist(),
    y=corr.index.tolist(),
    colorscale='RdBu_r',
    zmin=-1, zmax=1,
    text=corr.values,
    texttemplate='%{text:.2f}',
    textfont_size=15,
    hovertemplate='%{y} x %{x}: %{z:.2f}<extra></extra>',
    colorbar=dict(title='Pearson r'),
))
fig.update_layout(
    title_text='Correlation matrix – numerical features',
    height=430,
)
fig.show()

## Feature association with churn

**Cramér's V** measures association between a categorical feature and the binary target (range 0–1). **Absolute point-biserial correlation** does the same for numerical features. Both are plotted on a shared scale. This chart is the pre-modelling version of a SHAP feature-importance plot — the ranking here should largely match the mean absolute SHAP values after training.

In [ ]:
def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(ct)[0]
    n = ct.values.sum()
    r, k = ct.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))

target = (df_train['Churn'] == 'Yes').astype(int)

records = []
for col in cat_cols:
    records.append({'Feature': col, 'Score': cramers_v(df_train[col], df_train['Churn']), 'Method': "Cramer's V"})
for col in num_cols:
    r, _ = pointbiserialr(df_train[col].fillna(df_train[col].median()), target)
    records.append({'Feature': col, 'Score': abs(r), 'Method': '|Point-biserial|'})

assoc = pd.DataFrame(records).sort_values('Score')

fig = px.bar(
    assoc,
    x='Score', y='Feature',
    orientation='h',
    color='Method',
    color_discrete_map={"Cramer's V": RETAIN, '|Point-biserial|': CHURN},
    text=assoc['Score'].map('{:.3f}'.format),
    labels={'Score': 'Association strength', 'Feature': ''},
    title='Feature association with churn  |  Cramer\'s V (categorical)  vs  |point-biserial| (numerical)',
    height=620,
)
fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis_range=[0, assoc['Score'].max() * 1.25],
    legend_title='Method',
)
fig.show()

## Key findings

### Contract type
The single strongest signal (Cramér's V ≈ 0.40). Month-to-month customers churn at ~43 % versus ~3 % for two-year customers. The mechanism is commitment cost: longer contracts impose financial penalties and psychological inertia. In the Streamlit app, `Contract = Month-to-month` will be one of the most frequent positive SHAP contributors on a high-risk prediction.

### Tenure
The time dimension of the same effect. The first 12 months are critical: ~47 % of customers in that cohort churn. After year one, the rate halves; after year two, it drops to ~13 %. Customers who survive the initial trial period build habits and accept sunk costs. Short tenure combined with month-to-month contract is the highest-risk compound profile.

### Internet service — fibre optic
Fibre-optic subscribers churn at ~42 %, nearly twice the DSL rate. This is counter-intuitive — fibre is a premium product — but reflects that fibre customers pay significantly more per month, face more aggressive competitor offers in that segment, and are disproportionately on month-to-month contracts. High value does not imply high loyalty.

### Service bundling
Churn falls almost monotonically as add-on service count rises: 0 services → ~40 % churn; 5–6 services → under 10 %. Every additional service increases switching cost and deepens ecosystem dependency. In the SHAP chart, the *absence* of `TechSupport`, `OnlineSecurity`, or `OnlineBackup` will push the probability upward.

### Payment method and paperless billing
Electronic check + paperless billing is the highest-risk payment profile (~45 % churn). Automatic payment methods (credit card, bank transfer) correlate with ~15 % churn. The mechanism is engagement depth: customers who automate payments are more committed; those paying manually each cycle are actively re-evaluating the relationship.

### Senior citizens
Senior citizens churn at nearly twice the rate of non-seniors across all contract types. The gap is widest on month-to-month contracts. This likely reflects a combination of price sensitivity, lower service-complaint escalation rates, and exposure to competitor offers. `SeniorCitizen = 1` will reliably push the SHAP probability upward.

### Multicollinearity
`TotalCharges` is near-deterministically derived from `tenure × MonthlyCharges` (r = 0.83 with tenure). Retaining all three will dilute SHAP values by splitting importance across correlated features. The preprocessing step should drop `TotalCharges` or apply VIF-based selection.

### Modelling implications
- Use **ROC-AUC** and **F1 / precision-recall** as primary metrics, not accuracy.
- Apply `class_weight='balanced'` or `scale_pos_weight` to handle the 26 % churn imbalance.
- `Contract`, `tenure`, `InternetService`, `n_services`, and `PaymentMethod` are the five features most likely to dominate the SHAP waterfall on a high-risk customer.